Reinserting Trackers 0B - PV Paiva (pairs of 32p → 64p)
=
Works on `PV Paiva - Eletromecânico 2.dwg`, a version of the Paiva electromechanical drawing that already has the new `EMF_tracker_64p` / `EMF_tracker_32p` blocks. It pairs the `EMF_tracker_32p` blocks that sit in the same column and close to each other, and writes a script that replaces each pair with one `EMF_tracker_64p` and re-inserts the unpaired ones as `EMF_tracker_32p`. The `EMF_tracker_64p` blocks already in the drawing are not re-inserted.

**Workflow overview:**
1. Convert the DWG to DXF and extract all model space entities.
2. Look at the entity counts per block name.
3. Keep only the tracker blocks.
4. Define the pairing functions.
5. Pair the `EMF_tracker_32p` blocks.
6. Write the reinsert script.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `PV Paiva - Eletromecânico 2.dwg` in `C:\Users\Usuario\Desktop\projetos\paiva`. The DXF is written to `...\paiva\output`. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** trackers are block references (`INSERT`) placed directly in model space.
- **Block names** (exact, case-sensitive `==`): `EMF_tracker_64p` and `EMF_tracker_32p`. Only the `32p` ones are paired.
- **Attributes:** some block in the drawing must have the `QBT`, `DSJ`, `PVM`, `GRP`, `UFV`, `PST` and `INV` attributes (the inverters, in Paiva), because those columns are dropped by name. Without them the `drop` raises `KeyError`.
- **Geometry:** trackers run north-south. Blocks of the same column have the same X (within 0.1 after rounding to 2 decimals), and two `32p` blocks are paired when their Y gap is ≤ 44. The new `64p` goes at the upper block's Y − 21.264, which is specific to these block definitions.
- **Layers:** the new block goes on the layer of the upper block of the pair (`Layer-1`); the layer of the lower one is ignored.
- **Running the `.scr` in AutoCAD:** each line is a plain command-line `._insert <block> X,Y 1 1 0` (spaces act as Enter), so:
  - the blocks `EMF_tracker_64p` and `EMF_tracker_32p` must already be defined in the drawing;
  - they must **not** have *Scale uniformly* on, because the line answers X scale, Y scale and rotation;
  - if they have attributes, set `ATTREQ = 0`, because the script gives no attribute values;
  - everything is inserted with rotation 0.
  - delete the existing `EMF_tracker_32p` blocks first (not the `64p` ones), otherwise they end up duplicated.

Loading the DWG entities
=
- Copies `PV Paiva - Eletromecânico 2.dwg` from `dwg_folder` into `temp_folder`, so the ODA File Converter only processes that single file.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) converts it to DXF (ACAD2018) in ``output_folder` (`paiva\output`)`.
- `extract_data_from_dxf` reads every model space entity with `ezdxf` into a list of dicts (no DuckDB here). For block references it keeps the block name, insertion point, rotation, color and every attribute tag as its own key.
- Deletes the DXF and the temp copy of the DWG afterwards.
- `pd` comes from the `*` import of `DWG_to_DuckDB_ETL`. `math` is not used.

In [166]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico 2.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico 2.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico 2.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


Entity overview
=
Converts the entities into a DataFrame and prints `describe()`, `info()` and the count per block name. In the saved run: 4126 `EMF_tracker_64p`, 4111 `EMF_tracker_32p`, 464 inverters and 29 PSTs.

In [167]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y     rotation      center_x       center_y  \
count   8730.000000    8730.000000  8730.000000      1.000000       1.000000   
mean   35066.314587  128085.354872    10.731959  36799.390071  127830.355783   
std      593.385771     951.366577    42.569309           NaN            NaN   
min    33667.553490  125999.729475     0.000000  36799.390071  127830.355783   
25%    34626.886150  127315.224600     0.000000  36799.390071  127830.355783   
50%    35003.257868  127981.019775     0.000000  36799.390071  127830.355783   
75%    35496.286150  128973.818200     0.000000  36799.390071  127830.355783   
max    36625.891051  129766.459488   270.000000  36799.390071  127830.355783   

           radius  
count    1.000000  
mean   318.270393  
std           NaN  
min    318.270393  
25%    318.270393  
50%    318.270393  
75%    318.270393  
max    318.270393  
<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 17 columns):
 # 

Selecting the tracker blocks
=
- Keeps `EMF_tracker_64p` and `EMF_tracker_32p`.
- Drops the columns that are not needed, including `rotation` and the `UFV` / `PST` / `INV` attributes. Every block is re-inserted at rotation 0 and without tags.

In [168]:
mask = (ELM_df['name'] == 'EMF_tracker_64p') | (ELM_df['name'] == 'EMF_tracker_32p')
tracker_df = ELM_df[mask].copy()

tracker_df.drop(columns=['handle', 'type', 'QBT', 'DSJ', 'PVM', 'GRP', 'rotation', 'UFV', 'PST', 'INV'], inplace=True)

print(tracker_df)
print(tracker_df.describe())
print(tracker_df.info())

              layer             name             X              Y  center_x  \
522   EMF_em_pst_21  EMF_tracker_64p  34979.686150  127486.311575       NaN   
523   EMF_em_pst_21  EMF_tracker_64p  34985.986150  127486.311575       NaN   
524   EMF_em_pst_21  EMF_tracker_64p  34992.286150  127486.311575       NaN   
525   EMF_em_pst_21  EMF_tracker_64p  34973.386150  127486.311575       NaN   
526   EMF_em_pst_21  EMF_tracker_64p  35004.886150  127486.311575       NaN   
...             ...              ...           ...            ...       ...   
8755  EMF_em_pst_01  EMF_tracker_32p  35683.005250  129540.154750       NaN   
8756  EMF_em_pst_01  EMF_tracker_32p  35689.305250  129540.154750       NaN   
8757  EMF_em_pst_01  EMF_tracker_32p  35701.905250  129540.154750       NaN   
8758  EMF_em_pst_01  EMF_tracker_32p  35708.205250  129540.154750       NaN   
8759  EMF_em_pst_09  EMF_tracker_32p  34127.453648  128845.259495       NaN   

      center_y  radius  
522        NaN     NaN  
5

Pairing functions
=
**`group_structures(df, max_distance=44)`**
- Rounds `X` and `Y` to 2 decimals (this also changes the frame passed in) and sorts by `X` ascending, then `Y` descending: column by column, top to bottom. The docstring says "Y then X", which is not what the code does.
- `x_diff` / `y_diff` = difference to the previous block in that order.
- A new group starts when `x_diff > 0.1` (new column), when `|y_diff| > 44` (too far from the block above), or at the first row.
- Groups of 1 or 2 are kept as they are. Bigger groups are cut into consecutive pairs from the top, and an odd last block stays alone. The cut ignores the actual gaps inside the group.
- Notes:
  - `distance` is computed but not used.
  - The docstring mentions TAG boundaries and a 22 m default; neither is implemented (the default is 44), and there is no layer / PST check, so two blocks on different PST layers can be paired.
  - Every call writes the sorted frame to `Desktop\teste.xlsx` (debug export).

**`create_group_positions(groups)`**
- One row per group with the **first (upper)** block's `X` / `Y`, the group size and `Layer-1` (plus `Layer-2` for pairs).

In [169]:
def group_structures(df, max_distance=44):
    """
    Groups adjacent structures in pairs (2) while respecting:
    - TAG boundaries
    - Maximum distance constraint (22m default)
    - Sorted by Position Y then X
    """
    # Sort by Position Y (primary) and Position X (secondary)
    df['X'] = round(df['X'],2)
    df['Y'] = round(df['Y'],2)
    df = df.sort_values(['X', 'Y'], ascending=[True, False]).reset_index(drop=True)




    # Calculate pairwise distances between consecutive structures
    df['x_diff'] = round(df['X'].diff(),2)
    df['y_diff'] = round(df['Y'].diff(),2)
    df['distance'] = np.sqrt(df['x_diff']**2 + df['y_diff']**2)

    # Create group boundaries where:
    # 1. X is different
    # 2. Distance > max_distance OR
    # 3. It's the first structure
    df['group_boundary'] = (
        (df['x_diff'] > 0.1) |
        (abs(df['y_diff']) > max_distance) |
        (df.index == 0)
    )
    # Create group IDs
    df['group_id'] = df['group_boundary'].cumsum()

    # Split into groups
    result_groups = []

    for _, group_df in df.groupby('group_id'):
        # Split groups larger than 2 into pairs
        size = len(group_df)
        if size <= 2:
            result_groups.append(group_df)

            continue

        n_pairs = size // 2
        for i in range(n_pairs):
            result_groups.append(group_df.iloc[i*2:i*2+2])


        if size % 2:  # Handle odd-sized groups
            result_groups.append(group_df.iloc[-1:])
    df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")
    return result_groups

def create_group_positions(groups):
    """Create the output DataFrame from groups (same as before)"""
    group_data = []
    for i, group_df in enumerate(groups, 1):
        if len(group_df) > 1:
            first_struct = group_df.iloc[0]
            second_struct = group_df.iloc[1]
            group_data.append({
                'Group_ID': i,
                'X': first_struct['X'],
                'Y': first_struct['Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer']
            })
        else:
            first_struct = group_df.iloc[0]
            group_data.append({
                'Group_ID': i,
                'X': first_struct['X'],
                'Y': first_struct['Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer']
            })
    return pd.DataFrame(group_data)

Pairing the 32p blocks
=
- Runs the pairing on the `EMF_tracker_32p` blocks only.
- Exports the groups to `Desktop\teste1.xlsx` (debug) and prints the group sizes: 3407 single blocks and 352 pairs in the saved run.

In [170]:
mask_32p = (tracker_df['name'] == 'EMF_tracker_32p')
df_32p = tracker_df[mask_32p].copy()
results = group_structures(df_32p)

group_positions = create_group_positions(results)
group_positions.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")
print(group_positions['Group_Size'].value_counts())

Group_Size
1    3407
2     352
Name: count, dtype: int64


Reinsert script
=
- Single block → `._insert EMF_tracker_32p X,Y 1 1 0` at the same point.
- Pair → `._insert EMF_tracker_64p X,Y 1 1 0` at the upper block's `Y − 21.264` (`correction_position_y`).
- Exports the table to `...\Ecotechne\Separador de Strings\teste.xlsx` (debug; this is a different project folder).
- Writes the lines grouped by `Layer-1`, each group after `(command-s "._layer" "set" "<layer>" "")`, to `Desktop\Pessoal\Python\Ecotechne\Reinserting Trackers\Reinserting Trackers 0B.scr`.
  - Warning: the `Reinserting Trackers` notebook writes a file with the same name, so running one overwrites the other's script.

In [171]:
correction_position_y = 21.264

group_positions['Script'] = np.where(
    group_positions['Group_Size'] == 1,
    group_positions.apply(
        lambda row: (
            f"._insert EMF_tracker_32p {row['X']},{row['Y']} 1 1 0"
        ),
    axis=1
    ),
    group_positions.apply(
        lambda row: (
            f"._insert EMF_tracker_64p {row['X']},{row['Y']-correction_position_y} 1 1 0"
        ),
    axis=1
    )
)
print(group_positions)

group_positions.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\teste.xlsx")
with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Reinserting Trackers\Reinserting Trackers 0B.scr", 'w') as f:
    for i in group_positions['Layer-1'].unique():
        mask = group_positions['Layer-1'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(group_positions[mask]['Script'].astype(str)) + '\n')

      Group_ID         X          Y  Group_Size        Layer-1 Layer-2  \
0            1  33667.55  129034.80           1  EMF_em_pst_07     NaN   
1            2  33667.55  128974.79           1  EMF_em_pst_07     NaN   
2            3  33673.85  129034.80           1  EMF_em_pst_07     NaN   
3            4  33673.85  128974.79           1  EMF_em_pst_07     NaN   
4            5  33680.15  129034.80           1  EMF_em_pst_07     NaN   
...        ...       ...        ...         ...            ...     ...   
3754      3755  36600.69  129536.32           1  EMF_em_pst_03     NaN   
3755      3756  36606.99  129536.32           1  EMF_em_pst_03     NaN   
3756      3757  36613.29  129536.32           1  EMF_em_pst_03     NaN   
3757      3758  36619.59  129536.32           1  EMF_em_pst_03     NaN   
3758      3759  36625.89  129536.32           1  EMF_em_pst_03     NaN   

                                                 Script  
0      ._insert EMF_tracker_32p 33667.55,129034.8 1 1